# Lesson 2.6 — experiments: iterative deepening

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

The function `run` runs one search and returns the goal node and the number of expansions. The function `largest_frontier` measures the largest frontier; your code does not report it, so memory is always measured on the course's reference versions. BFS comes from the reference version (lesson 2.2). The function `tree(b, d)` builds a full tree with branching factor `b` and depth `d`, with the goal in the deepest leaf on the right, the place DFS reaches last.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search import CUTOFF, reference
from introai.search.examples import complete_tree
from introai.search.problems import GraphProblem
from introai.testing.search import max_frontier
from introai.trace import TraceLimitExceeded, record
from introai.world.warehouse import RouteProblem, WarehouseMap

MAPS = Path.cwd().parents[2] / "data" / "maps"
HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'


def run(search, *args, limit=200_000):
    """Run one search; return its result and the number of expansions."""
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            result = search(*args)
    except NotImplementedError:
        raise RuntimeError(f"{search.__name__} is not written yet - {HINT}") from None
    except (TraceLimitExceeded, RecursionError):
        raise RuntimeError(f"{search.__name__} expanded more than {limit} nodes - {HINT}") from None
    return result, len(trace.of_kind("expand"))


def largest_frontier(search, problem):
    with record(kinds={"frontier"}) as trace:
        search(problem)
    return max_frontier(trace)


def tree(b, d):
    adjacency = complete_tree(b, d).adjacency
    return GraphProblem(adjacency, "r", "r" + str(b - 1) * d)

## Experiment 1: full trees

We compare BFS and IDS on full trees with branching factor 2, 3 and 4 and depth from 2 to 8.

**Your prediction.** How many times more nodes will IDS expand than BFS for *b* = 2, and how many for *b* = 4? How many nodes does each of them keep in the frontier at once for *b* = 3 and *d* = 8?

*My prediction:* …

In [ ]:
depths = range(2, 9)
ratios = {}
memory = {"BFS": [], "IDS": []}
print(f"{'b':>2} {'d':>2} {'BFS':>7} {'IDS':>7} {'IDS/BFS':>8} {'mem BFS':>8} {'mem IDS':>8}")
for b in (2, 3, 4):
    ratios[b] = []
    for d in depths:
        _, bfs_work = run(reference.bfs, tree(b, d))
        found, ids_work = run(algorithms.iterative_deepening_search, tree(b, d))
        if found is None or found == CUTOFF:
            raise RuntimeError(
                f"iterative deepening did not find the goal of tree({b}, {d}) - {HINT}"
            )
        ratios[b].append(ids_work / bfs_work)
        bfs_memory = largest_frontier(reference.bfs, tree(b, d))
        ids_memory = largest_frontier(reference.iterative_deepening_search, tree(b, d))
        if b == 3:
            memory["BFS"].append(bfs_memory)
            memory["IDS"].append(ids_memory)
        print(
            f"{b:>2} {d:>2} {bfs_work:>7} {ids_work:>7} {ids_work / bfs_work:>8.2f} "
            f"{bfs_memory:>8} {ids_memory:>8}"
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for b, color in zip((2, 3, 4), ("#2a78d6", "#eb6834", "#1baf7a")):
    ax.plot(
        list(depths),
        ratios[b],
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=f"b = {b}",
    )
    ax.annotate(
        f"b = {b}",
        (depths[-1], ratios[b][-1]),
        textcoords="offset points",
        xytext=(8, 0),
        va="center",
        color="#0b0b0b",
    )
ax.set_title("Expansions of IDS divided by expansions of BFS", loc="left", color="#0b0b0b")
ax.set_xlabel("depth of the goal", color="#52514e")
ax.margins(x=0.12)
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")

ax = axes[1]
for label, color in (("BFS", "#2a78d6"), ("IDS", "#eb6834")):
    ax.plot(
        list(depths),
        memory[label],
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
ax.set_yscale("log")
ax.set_title("Largest frontier for b = 3 (log scale)", loc="left", color="#0b0b0b")
ax.set_xlabel("depth of the goal", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Think about it.** Which values do the work ratios approach for *b* = 2, 3 and 4? Compare them with the formula *b*/(*b* − 1). How does the BFS frontier grow, and how does the IDS frontier grow, when the depth goes up by 1?

## Experiment 2: the warehouse

Now the same algorithms on warehouse maps, where every step costs 1 and very many paths lead to most cells.

**Your prediction.** Will IDS expand about twice as many cells as BFS on the map with aisles, as on a tree? And on the open floor?

*My prediction:* …

In [ ]:
names = ["small", "aisles", "open_floor", "depot"]
per_limit = {}
for name in names:
    warehouse = WarehouseMap.from_file(MAPS / f"{name}.txt")
    bfs_node, bfs_work = run(reference.bfs, RouteProblem(warehouse))
    counts, limit = [], 0
    while True:
        result, work = run(algorithms.depth_limited_search, RouteProblem(warehouse), limit)
        counts.append(work)
        if result != CUTOFF:
            break
        limit += 1
    per_limit[name] = counts
    print(
        f"{name:>10}: route {len(bfs_node.solution())} steps, BFS expanded {bfs_work}, "
        f"IDS expanded {sum(counts)} in {len(counts)} rounds"
    )

fig, ax = plt.subplots(figsize=(8, 4))
for name, color in zip(("aisles", "open_floor"), ("#2a78d6", "#eb6834")):
    counts = per_limit[name]
    ax.plot(
        range(1, len(counts)),
        counts[1:],  # limit 0 expands nothing, which a log scale cannot show
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=name,
    )
ax.set_yscale("log")
ax.set_title("Nodes expanded by depth-limited search, by limit", loc="left", color="#0b0b0b")
ax.set_xlabel("depth limit", color="#52514e")
ax.set_ylabel("nodes expanded (log scale)", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Think about it.** Why does IDS lose to BFS much more clearly on the maps than on a tree? Recall lesson 2.1: what does IDS remember and what not, and how many different paths lead to one cell on the open floor? When, then, is IDS worth using?

## Summary

Go back to the review questions in the lesson. After your own attempt, you can compare your results and conclusions with the answers in `solutions/02_search/06_iterative_deepening/answers.md`.